# Tennis Vision — Colab Runner
Analyzes a tennis video: ball tracking, player detection, pose, speed & in/out calls.

---
## Sections
- **[A] Setup** — clone, install, download models (run once)
- **[B] Process Video** — analyze your YouTube Shorts video
- **[C] Train TrackNet** — best ball detector, uses 3 frames (recommended)
- **[D] Fine-tune YOLOv8 Ball Detector** — alternative single-frame detector
- **[E] Fine-tune Court Detector** — improve in/out & speed accuracy
- **[F] Save/Load Models** — persist trained models to Google Drive
---
> **First time?** Run Section A top to bottom, then B.
> **Already set up?** Just run Cell A1 (`git pull`) then go to B.
> **Best accuracy order:** A → C (TrackNet) → E (Court) → B

## Section A — Setup

In [ ]:
# ── A1: Clone repo (skips if already cloned, pulls latest if exists) ────────
import os

if not os.path.exists('/content/tennis_vision'):
    !git clone https://github.com/bozanctn/tennis_vision.git /content/tennis_vision
    print('Cloned.')
else:
    print('Repo already exists — pulling latest changes...')
    !git -C /content/tennis_vision pull origin main

%cd /content/tennis_vision

In [ ]:
# ── A2: Install dependencies ─────────────────────────────────────────────────
!pip install -q -r requirements.txt
!pip install -q roboflow
print('\nAll dependencies installed.')

In [ ]:
# ── A3: Download base YOLOv8 model ───────────────────────────────────────────
!python scripts/download_models.py

## Section B — Process Your Video

In [ ]:
# ── B1: Process YouTube Shorts video ─────────────────────────────────────────
YOUTUBE_URL = 'https://www.youtube.com/shorts/08-cKvAJMVo'
OUTPUT_FILE = '/content/tennis_vision/result.mp4'

!python scripts/process_video.py \
    --input "{YOUTUBE_URL}" \
    --output "{OUTPUT_FILE}" \
    --device cuda

print(f'\nDone! Output saved to: {OUTPUT_FILE}')

In [ ]:
# ── B2: Preview result inside Colab ──────────────────────────────────────────
from IPython.display import HTML
from base64 import b64encode
import os

output_file = '/content/tennis_vision/result.mp4'

if os.path.exists(output_file):
    playable = output_file.replace('.mp4', '_preview.mp4')
    !ffmpeg -y -i "{output_file}" -vcodec libx264 -acodec aac "{playable}" -loglevel quiet
    with open(playable, 'rb') as f:
        video_data = f.read()
    b64 = b64encode(video_data).decode()
    display(HTML(f'''
        <video width="720" controls>
            <source src="data:video/mp4;base64,{b64}" type="video/mp4">
        </video>
    '''))
else:
    print('Output file not found — check B1 for errors.')

In [ ]:
# ── B3: Download result to your computer ─────────────────────────────────────
from google.colab import files
files.download('/content/tennis_vision/result.mp4')

In [ ]:
# ── B4 (optional): Save to Google Drive ──────────────────────────────────────
from google.colab import drive
import shutil
drive.mount('/content/drive')
shutil.copy('/content/tennis_vision/result.mp4', '/content/drive/MyDrive/tennis_result.mp4')
print('Saved to Google Drive: tennis_result.mp4')

## Section C — Train TrackNet Ball Detector (Recommended)

**TrackNet is the best approach for tennis ball detection.**
Unlike YOLOv8 (single frame), TrackNet sees 3 consecutive frames at once,
so it can detect the ball even with heavy motion blur.

```
YOLOv8:    frame_t             → bounding box
TrackNet:  frame_t-2, t-1, t  → heatmap (much better for fast balls)
```

**What you need:**
1. A free Roboflow API key → [roboflow.com](https://roboflow.com) → Settings → API
2. That's it — dataset download, labeling prep, training all automatic

In [ ]:
# ── C1: Set your Roboflow API key ────────────────────────────────────────────
ROBOFLOW_API_KEY = 'PASTE_YOUR_KEY_HERE'   # <-- paste here

In [ ]:
# ── C2: Download tennis ball dataset + convert to TrackNet CSV format ─────────
# Downloads a public dataset with ball position annotations per frame
# Then converts it to the CSV format TrackNet training expects:
#   video_path, frame_idx, cx, cy

from roboflow import Roboflow
import os, csv, json
from pathlib import Path

os.makedirs('/content/tennis_vision/data', exist_ok=True)
os.chdir('/content/tennis_vision/data')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Download as COCO JSON format so we get per-frame ball coordinates
project = rf.workspace('tennis-nimqk').project('tennis-ball-detection-dkqrp')
dataset = project.version(1).download('coco')

BALL_DATASET_PATH = dataset.location
print(f'Dataset downloaded to: {BALL_DATASET_PATH}')

os.chdir('/content/tennis_vision')

# Convert COCO annotations → TrackNet CSV
# Each image is treated as an individual frame (frame_idx = image order)
csv_rows = []
for split in ['train', 'valid']:
    ann_file = Path(BALL_DATASET_PATH) / split / '_annotations.coco.json'
    if not ann_file.exists():
        continue
    with open(ann_file) as f:
        coco = json.load(f)

    # Map image_id → file_name
    id_to_img = {img['id']: img for img in coco['images']}
    # Map image_id → ball center (cx, cy)
    id_to_ball = {}
    for ann in coco['annotations']:
        x, y, w, h = ann['bbox']
        id_to_ball[ann['image_id']] = (x + w/2, y + h/2)

    for i, img_info in enumerate(coco['images']):
        img_path = str(Path(BALL_DATASET_PATH) / split / 'images' / img_info['file_name'])
        ball = id_to_ball.get(img_info['id'])
        cx = ball[0] if ball else ''
        cy = ball[1] if ball else ''
        csv_rows.append({'video_path': img_path, 'frame_idx': i, 'cx': cx, 'cy': cy})

csv_path = '/content/tennis_vision/data/tracknet_labels.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['video_path', 'frame_idx', 'cx', 'cy'])
    writer.writeheader()
    writer.writerows(csv_rows)

print(f'\nConverted {len(csv_rows)} frames to TrackNet CSV: {csv_path}')

In [ ]:
# ── C3: Train TrackNet (~20-30 min on T4 GPU) ─────────────────────────────────
# Trains from scratch on the tennis ball dataset
# Output: models/tracknet.pt  (pipeline auto-uses this if it exists)

!python scripts/train_tracknet.py \
    --csv /content/tennis_vision/data/tracknet_labels.csv \
    --epochs 30 \
    --batch_size 4 \
    --output /content/tennis_vision/models/tracknet.pt \
    --device cuda

In [ ]:
# ── C4: Verify TrackNet is being used ────────────────────────────────────────
import sys
sys.path.insert(0, '/content/tennis_vision')
import yaml
from src.detection.ball_detector import BallDetector

with open('/content/tennis_vision/config/config.yaml') as f:
    cfg = yaml.safe_load(f)

# Temporarily point tracknet path to our trained model
cfg['models']['tracknet_model_path'] = 'models/tracknet.pt'
with open('/content/tennis_vision/config/config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

detector = BallDetector.from_config(cfg)
print(f'Active backend: {detector.backend}')  # should print: tracknet
print('TrackNet is ready! Re-run Section B to process your video.')

## Section D — Fine-tune YOLOv8 Ball Detector (Alternative)

Use this if you want a single-frame detector (simpler, faster, less accurate than TrackNet).
Skip if you already trained TrackNet in Section C.

In [ ]:
# ── D1: Download dataset + train YOLOv8 ball detector ────────────────────────
from roboflow import Roboflow
import os

os.makedirs('/content/tennis_vision/data', exist_ok=True)
os.chdir('/content/tennis_vision/data')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('tennis-nimqk').project('tennis-ball-detection-dkqrp')
dataset = project.version(1).download('yolov8')
BALL_DATASET_PATH = dataset.location
os.chdir('/content/tennis_vision')

!yolo train \
  model=yolov8n.pt \
  data="{BALL_DATASET_PATH}/data.yaml" \
  epochs=50 \
  imgsz=640 \
  batch=16 \
  project=/content/tennis_vision/models \
  name=ball_detector \
  device=0 \
  exist_ok=True

In [ ]:
# ── D2: Check results + update config ────────────────────────────────────────
from ultralytics import YOLO
from IPython.display import Image, display
import os, yaml

best_weights = '/content/tennis_vision/models/ball_detector/weights/best.pt'
if os.path.exists(best_weights):
    model = YOLO(best_weights)
    metrics = model.val(data=f'{BALL_DATASET_PATH}/data.yaml', device=0)
    print(f'mAP@50: {metrics.box.map50:.3f}  (aim >0.70)')

    results_img = '/content/tennis_vision/models/ball_detector/results.png'
    if os.path.exists(results_img):
        display(Image(results_img, width=900))

    config_path = '/content/tennis_vision/config/config.yaml'
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    cfg['models']['ball_model_path'] = 'models/ball_detector/weights/best.pt'
    with open(config_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print('Config updated — re-run Section B!')

## Section E — Fine-tune Court Detector

Improves court keypoint accuracy → better in/out calls & more accurate speed.

In [ ]:
# ── E1: Download court dataset + train ───────────────────────────────────────
from roboflow import Roboflow
import os

os.makedirs('/content/tennis_vision/data', exist_ok=True)
os.chdir('/content/tennis_vision/data')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('tennis-court-zqmfm').project('tennis-court-detection-uq5vc')
court_dataset = project.version(1).download('yolov8')
COURT_DATASET_PATH = court_dataset.location
os.chdir('/content/tennis_vision')

!yolo train \
  model=yolov8n.pt \
  data="{COURT_DATASET_PATH}/data.yaml" \
  epochs=80 \
  imgsz=1280 \
  batch=8 \
  project=/content/tennis_vision/models \
  name=court_detector \
  device=0 \
  exist_ok=True

In [ ]:
# ── E2: Update config to use fine-tuned court model ──────────────────────────
import yaml

config_path = '/content/tennis_vision/config/config.yaml'
with open(config_path) as f:
    cfg = yaml.safe_load(f)
cfg['models']['court_model_path'] = 'models/court_detector/weights/best.pt'
with open(config_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print('Done — re-run Section B!')

## Section F — Save / Load Models (Google Drive)

Colab resets every session — save trained models to Drive once, load them next time.

In [ ]:
# ── F1: Save all trained models to Google Drive ───────────────────────────────
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
drive_dir = '/content/drive/MyDrive/tennis_vision_models'
os.makedirs(drive_dir, exist_ok=True)

to_save = {
    '/content/tennis_vision/models/tracknet.pt':                        'tracknet.pt',
    '/content/tennis_vision/models/ball_detector/weights/best.pt':      'ball_detector.pt',
    '/content/tennis_vision/models/court_detector/weights/best.pt':     'court_detector.pt',
}
for src, name in to_save.items():
    if os.path.exists(src):
        shutil.copy(src, f'{drive_dir}/{name}')
        print(f'Saved {name}')
    else:
        print(f'Skipped {name} (not found)')

print(f'\nAll saved to: {drive_dir}')

In [ ]:
# ── F2: Load models from Drive in a new session ───────────────────────────────
from google.colab import drive
import shutil, os, yaml

drive.mount('/content/drive')
drive_dir  = '/content/drive/MyDrive/tennis_vision_models'
local_dir  = '/content/tennis_vision/models'
os.makedirs(local_dir, exist_ok=True)

to_load = ['tracknet.pt', 'ball_detector.pt', 'court_detector.pt']
for name in to_load:
    src = f'{drive_dir}/{name}'
    if os.path.exists(src):
        shutil.copy(src, f'{local_dir}/{name}')
        print(f'Loaded {name}')

# Update config
config_path = '/content/tennis_vision/config/config.yaml'
with open(config_path) as f:
    cfg = yaml.safe_load(f)

if os.path.exists(f'{local_dir}/tracknet.pt'):
    cfg['models']['tracknet_model_path'] = 'models/tracknet.pt'
if os.path.exists(f'{local_dir}/ball_detector.pt'):
    cfg['models']['ball_model_path']     = 'models/ball_detector.pt'
if os.path.exists(f'{local_dir}/court_detector.pt'):
    cfg['models']['court_model_path']    = 'models/court_detector.pt'

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('\nConfig updated — run Section B!')